<a href="https://colab.research.google.com/github/RushikeshBawaskar/fine_tuning/blob/main/non_Instruction_pretrain_llm_finetuning_on_domain_specific_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U peft bitsandbytes transformers accelerate

In [ ]:
!pip install -U trl

In [ ]:
!pip install PyMuPDF

In [ ]:
from datasets import Dataset, load_dataset

In [ ]:
import fitz

In [ ]:
def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

In [ ]:
pdf_texts = extract_text_from_pdf("/content/Metformin.pdf")

In [ ]:
pdf_texts

In [ ]:
import re
def split_paragraphs(pages):
    paragraphs = []
    for page_text in pages:
        # Split on double line breaks or long newlines
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30:  # ignore too short lines
                paragraphs.append(clean)
    return paragraphs


In [ ]:
paragraphs = split_paragraphs(pdf_texts)

In [ ]:
data = [{"text": p} for p in paragraphs]

In [ ]:
dataset = Dataset.from_list(data)

In [ ]:
dataset

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(examples):
    tokens = tokenizer(examples["text"],truncation=True,padding="max_length",max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [ ]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

In [ ]:
tokenized

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
training_args = TrainingArguments(
    output_dir="./llama-pharma-domain",
    # overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
)

In [ ]:
trainer.train()

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!pip install -U peft bitsandbytes transformers accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [ ]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [ ]:
tokenized = dataset.map(tokenize_fn, batched=True)

In [ ]:
print(tokenized[0]['text'])
print("=======")
print(tokenized[0]['input_ids'])
print("=======")
print(tokenized[0]['attention_mask'])
print("=======")
print(tokenized[0]['labels'])
print("=======")

In [ ]:
# 1. Create the configuration for 8-bit loading
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model,
    quantization_config=quantization_config,
    device_map="auto"
)
print(f"Model is on: {model.device}")

In [ ]:
from peft import prepare_model_for_kbit_training

# This prepares the 8-bit model for training
model = prepare_model_for_kbit_training(model)

In [ ]:

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [ ]:
non_inst_model_lora = get_peft_model(model, lora_config)

In [ ]:
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=non_inst_model_lora,
    args=args,
    train_dataset=tokenized
)

In [ ]:
print(f"Model is on: {non_inst_model_lora.device}")

In [ ]:
import torch
print(torch.cuda.is_available())
# This MUST print True. If it says False, the GPU isn't active.


trainer.train()

In [ ]:
model_path = "/content/tinyllama-lora/checkpoint-5"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

In [ ]:
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
import json

def fix_notebook_metadata(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    # Remove the problematic widgets metadata
    if 'widgets' in nb.get('metadata', {}):
        del nb['metadata']['widgets']
        print("Removed metadata.widgets")
    else:
        print("No metadata.widgets found.")

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=1)
    print(f"Fixed notebook saved as: {output_file}")

# Note: This requires you to have the .ipynb file path.
# Since you are in a live session, you can export the current session's state
# or simply use the 'File' > 'Download' > '.ipynb' and then run a local script.
# However, usually just clearing cell outputs before saving to GitHub fixes this.
print("Tip: Try 'Edit' -> 'Clear all outputs' and then saving to GitHub. If that fails, use the script above on a downloaded file.")